Pydantic- Pydantic is a data validation library in python.

In [ ]:
from rich.pretty import pprint

In [ ]:
user_attributes = {
    "name": "Keanu Reeves",
    "age": 873, # because we all know Keanu Reeves is immortal
    "email": "jwick@email.com",
    "pets": ["dog"]
}

Using dataclasses
One way to turn our user input into an object is to use a dataclass.

In [ ]:
from dataclasses import dataclass

@dataclass
class User:
    name: str
    age: int
    email: str
    pets: list[str]

new_user = User(**user_attributes)
print(type(new_user))
pprint(new_user, expand_all=True)

def is_over_18(user: dict) -> bool:
    return user.age >= 18

is_over_18(new_user)

write some manual checks for typing in our age-checking function, but ideally we would like to keep all of the validation stuff in one place.

This is where Pydantic comes in.

In [ ]:
from pydantic import BaseModel, Field

class User(BaseModel):
    name: str = Field(..., title="Name", description="Name of the user")
    age: int = Field(..., title="Age", description="Age of the user")
    email: str = Field(..., title="Email", description="Email of the user")
    pets: list[str] = Field(..., title="Pets", description="Pets of the user")

user = User(**user_attributes)
pprint(user, expand_all=True)

is_over_18(user)

In [ ]:
user_attributes["age"] = "panda"

try:
    user = User(**user_attributes)
except Exception as e:
    print(e)

In [ ]:
class User(BaseModel):
    name: str = Field(..., title="Name", description="Name of the user")
    age: int = Field(..., title="Age", description="Age of the user", ge=18)
    email: str = Field(..., title="Email", description="Email of the user")
    pets: list[str] = Field(..., title="Pets", description="Pets of the user")

We have passed in the extra argument ge=18 to the Field class. This means that the age must be greater than or equal to 18. If we pass in an age less than 18, Pydantic will raise an error:

In [ ]:
from pydantic import ValidationError

user_attributes["age"] = 17

try:
    user = User(**user_attributes)
    pprint(user)
except ValidationError as e:
    pprint(e.errors())

This is handy, because it has told us what failure was, what we input, and what control we put in place to prevent it.

We can also define our own custom validators. Suppose we want to make sure that the email address is a valid email address. We can do this by defining a custom validator.

from pydantic import field_validator
from pydantic_core import PydanticCustomError

class User(BaseModel):
    name: str = Field(..., title="Name", description="Name of the user")
    age: int = Field(..., title="Age", description="Age of the user", ge=18)
    email: str = Field(..., title="Email", description="Email of the user")
    pets: list[str] = Field(..., title="Summary", description="Pets of the user")

    @field_validator("email")
    def check_email(cls, v):
        if "@" not in v or "." not in v:
            raise PydanticCustomError(
                'InvalidEmail',
                'Email must contain "@" and "."'
            )
        return v

In [ ]:
user_attributes = {
    "name": "Keanu Reeves",
    "age": 873,
    "email": "jwick-at-email-dot-com",
    "pets": ["dog"]
}

try:
    user = User(**user_attributes)
    pprint(user)
except ValidationError as e:
    pprint(e.errors(), expand_all=True)

Application to LLMs

An important application of Pydantic is in validating the output of LLMs.

In [ ]:
description = (
    "My name is Ryan, and I am 35 years old. "
    "During the weekends I like to hike, but I also enjoy playing video games. "
    "It can sometimes be difficult to use my computer, "
    "because my cat likes to sleep on the keyboard! "
    "During the week, I work as a MLE at the University of Cambridge. "
    "Although I really enjoy living in the UK, "
    "I miss the outdoors back home in NZ."
)

system_prompt = (
    "Your main role is to analyse a piece of unstructured text and extract the following information:\n"
    "- Name\n"
    "- Age\n"
    "- Nationality\n"
    "- Occupation\n"
    "- A list of any pets\n"
    "- A list of any hobbies\n\n"
    "If any acronyms are used, please expand them.\n\n"
    "Here is a description:\n\n"
)


In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": description},
    ],
    temperature=0.0,
)

print(response.choices[0].message.content)

I can try to get the output in a structured format.

First we define a new Pydantic class that we want the output to be.

In [ ]:
class Person(BaseModel):
    name: str | None = Field(..., description="The name of the person")
    age: int | None = Field(..., description="The age of the person")
    nationality: str | None = Field(..., description="The nationality of the person")
    occupation: str | None = Field(..., description="The occupation of the person")
    pets: list[str] | None = Field(..., description="The pets of the person")
    hobbies: list[str] | None = Field(..., description="The hobbies of the person")

    # print the description of the person
    def __str__(self) -> str:
        output = f"Name: {self.name}\n"
        output += f"Age: {self.age}\n"
        output += f"Nationality: {self.nationality}\n"
        output += f"Occupation: {self.occupation}\n"
        output += f"Pets: {self.pets}\n"
        output += f"Hobbies: {self.hobbies}\n"
        return output

In [ ]:
system_prompt = (
    "Your main role is to analyse a piece of unstructured text and extract the following information:\n"
    "- Name\n"
    "- Age\n"
    "- Nationality\n"
    "- Occupation\n"
    "- A list of any pets\n"
    "- A list of any hobbies\n\n"
    "If any acronyms are used, please expand them.\n"
    "Return the information in JSON format.\n\n" # <--- JSON please!
    "Here is a description:\n\n"
)

response = client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": description},
  ],
  max_tokens=512,
  temperature=0.0,
)

print(response.choices[0].message.content)

In [ ]:
system_prompt = (
    "Your main role is to analyse a piece of unstructured text and extract the following information:\n"
    "- Name\n"
    "- Age\n"
    "- Nationality\n"
    "- Occupation\n"
    "- A list of any pets\n"
    "- A list of any hobbies\n\n"
    "If any acronyms are used, please expand them.\n"
    "Return the information in JSON format. "
    "Do not include the `json` tags.\n\n" # <--- no tags please!
    "Here is a description:\n\n"
)

response = client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": description},
  ],
  max_tokens=512,
  temperature=0.0,
)

print(response.choices[0].message.content)

In [ ]:
system_prompt = (
    "Your main role is to analyse a piece of unstructured text and extract the following information:\n"
    "- Name\n"
    "- Age\n"
    "- Nationality\n"
    "- Occupation\n"
    "- A list of any pets\n"
    "- A list of any hobbies\n\n"
    "If any acronyms are used, please expand them.\n"
    "Return the information in JSON format.\n\n" # <--- JSON please!
    "Here is a description:\n\n"
)

response = client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": description},
  ],
  max_tokens=512,
  temperature=0.0,
  response_format={"type": "json_object"} # <--- JSON please!
)

print(response.choices[0].message.content)

In [ ]:
schema = Person.model_json_schema()
print(type(schema))
pprint(schema)

In [ ]:
system_prompt = (
    "Your main role is to analyse a piece of unstructured text and extract the following information:\n"
    "- Name\n"
    "- Age\n"
    "- Nationality\n"
    "- Occupation\n"
    "- A list of any pets\n"
    "- A list of any hobbies\n\n"
    "If any acronyms are used, please expand them.\n\n"
    f"Return the information in JSON format according to the following schema:\n\n{schema}\n\n"
    "Here is a description:\n\n"
)

print(system_prompt)

response = client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": description},
  ],
  max_tokens=512,
  response_format={"type": "json_object"},
  temperature=0.0
)

print(response.choices[0].message.content)